<a href="https://colab.research.google.com/github/Aryansahu9508/1835021MP3/blob/main/shootingmethod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import numpy as np

# Boundary conditions
a = 0
b = 1
h = 0.01

ya = 0
za = 0

yb = 1
zb = 0

N = int((b-a)/h)

# Coupled differential equations
# y'' = -y + z
# z'' = y - z

def equations(Y):
    y, u, z, v = Y

    return np.array([
        u,
        -y + z,
        v,
        y - z
    ])

# RK4 method
def RK4(s1, s2):

    Y = np.zeros((N+1, 4))

    Y[0] = [ya, s1, za, s2]

    for i in range(N):

        k1 = h * equations(Y[i])
        k2 = h * equations(Y[i] + k1/2)
        k3 = h * equations(Y[i] + k2/2)
        k4 = h * equations(Y[i] + k3)

        Y[i+1] = Y[i] + (
            k1 + 2*k2 + 2*k3 + k4
        )/6

    return Y

# Shooting method using Newton iteration
s1, s2 = 1.0, 0.0

for iteration in range(20):

    Y = RK4(s1, s2)

    F = np.array([
        Y[-1, 0] - yb,
        Y[-1, 2] - zb
    ])

    if np.linalg.norm(F) < 1e-10:
        break

    # Numerical Jacobian
    delta = 1e-5

    J = np.zeros((2, 2))

    for j in range(2):

        ds = np.zeros(2)
        ds[j] = delta

        Ynew = RK4(s1 + ds[0], s2 + ds[1])

        J[:, j] = (
            np.array([
                Ynew[-1, 0] - yb,
                Ynew[-1, 2] - zb
            ]) - F
        ) / delta

    correction = np.linalg.solve(J, F)

    s1 -= correction[0]
    s2 -= correction[1]

# Final result
Y = RK4(s1, s2)

print("Initial slopes:")
print("y'(0) =", s1)
print("z'(0) =", s2)

print("\nFinal boundary values:")
print("y(1) =", Y[-1, 0])
print("z(1) =", Y[-1, 2])

print("\nNumerical solution:")
print("    x        y          z")

for i in range(0, N+1, 10):
    print(
        round(a+i*h, 2),
        round(Y[i, 0], 6),
        round(Y[i, 2], 6)
    )


Initial slopes:
y'(0) = 1.215864708771517
z'(0) = -0.21586470875354305

Final boundary values:
y(1) = 1.0000000000164386
z(1) = 1.5358348273708522e-12

Numerical solution:
    x        y          z
0.0 0.0 0.0
0.1 0.121348 -0.021348
0.2 0.241272 -0.041272
0.3 0.358374 -0.058374
0.4 0.471317 -0.071317
0.5 0.578842 -0.078842
0.6 0.679801 -0.079801
0.7 0.773176 -0.073176
0.8 0.858103 -0.058103
0.9 0.933882 -0.033882
1.0 1.0 0.0
